## Repository path configuration
All repository files are accessed through relative paths. Run the notebook from the repository root or from the `notebooks/` directory.


In [ ]:
from pathlib import Path
import os

# Resolve the repository root whether Jupyter starts in the repository root
# or directly inside the notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

DATA_DIR = ROOT / "data"
EXTERNAL_DATA_DIR = DATA_DIR / "external"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = ROOT / "outputs"
RESULTS_DIR = OUTPUTS_DIR / "results"
FIGURES_DIR = OUTPUTS_DIR / "figures"
EXTERNAL_MATERIALS_DIR = ROOT / "external_materials"
MODEL_WEIGHTS_DIR = EXTERNAL_MATERIALS_DIR / "model_weights"

for folder in [
    EXTERNAL_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROCESSED_DATA_DIR / "logs",
    RESULTS_DIR,
    FIGURES_DIR,
    MODEL_WEIGHTS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)


### Optional Google Colab use
Google Drive mounting is not required. In Colab, clone or upload the complete repository and run the notebook from the repository root. External model weights, when needed, must be placed under `external_materials/model_weights/`.


In [ ]:
!pip install -U bitsandbytes

In [ ]:
import pandas as pd
import re
import os, random
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import shap
from tqdm import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig
)
from peft import get_peft_model, LoraConfig, TaskType

# Ενεργοποίηση του tqdm για τα pandas
tqdm.pandas()

In [ ]:
# ------------------------------------------------
# 1. Φόρτωση Master CSV
# ------------------------------------------------
MASTER_PATH = "data/processed/Master_results.csv"

df = pd.read_csv(MASTER_PATH)
print("Loaded Master_results.csv with shape:", df.shape)

Επειδή θα χρησιμοποιήσω και feature "is_citation", το παίρνω απο το αρχικό dataset που περιλαμβάνει γενικές προτάσεις και citations.
Έτσι γλιτώνω δημιουργία κανονικών εκφράσεων.

In [ ]:
PATH = "data/external/sentences_dataset_45269.csv"  # Το αρχικό dataset μου.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

df2 = pd.read_csv(PATH)

# Αφαίρεση διπλότυπων εγγραφών. Διατηρείται μόνο η πρώτη εμφάνιση της εν λόγω γραμμής.
# Υπολογισμός αριθμού διπλών εγγραφών.
num_duplicates = df2['sentence'].duplicated().sum()
print("Πλήθος διπλών εγγραφών στη στήλη sentence:", num_duplicates)

# Αποθήκευση του αρχικού πλήθους γραμμών.
before = len(df2)

# Αφαίρεση διπλότυπων (κρατάμε την πρώτη εμφάνιση).
df2 = df2.drop_duplicates(subset='sentence', keep='first')

# Υπολογισμός του πόσες γραμμές διαγράφηκαν.
after = len(df2)
deleted = before - after
print("Πλήθος εγγραφών που διαγράφηκαν:", deleted)

# ίδιο split όπως στα περισσότερα scripts (30% temp, 50-50 val/test)
train_data, temp_data = train_test_split(
    df2, test_size=0.3, stratify=df2["polarity"], random_state=SEED
)
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, stratify=temp_data["polarity"], random_state=SEED
)
print("Train:", train_data.shape, "Val:", val_data.shape, "Test:", test_data.shape)  # Με ενδιαφέρει το test.

# Έλεγχος αν υπάρχει ήδη η στήλη
if "is_citation" not in df.columns:
    print("Η στήλη is_citation δεν υπάρχει — γίνεται merge από το test_data")

    # Κρατάμε μόνο τις απαραίτητες στήλες
    test_subset = test_data[["sentence", "is_citation"]]

    # Merge με βάση το sentence
    df = df.merge(
        test_subset,
        on="sentence",
        how="left"
    )

    # Μετακίνηση στήλης δίπλα στη sentence.
    cols = list(df.columns)
    sentence_idx = cols.index("sentence")

    cols.insert(sentence_idx + 1, cols.pop(cols.index("is_citation")))
    df = df[cols]

    print("Merge ολοκληρώθηκε.")
else:
    print("Η στήλη is_citation υπάρχει ήδη — δεν έγινε merge.")

In [ ]:
num_citations = df[df['is_citation'] == 1].shape[0]
print(f"Ο αριθμός των εγγραφών που είναι citations: {num_citations}")

Προσθήκη feature θέσης αναφοράς. Για να βρούμε τη θέση μιας αναφοράς με επιστημονική ακρίβεια, χρησιμοποιούμε μια τεχνική που ονομάζεται Regular Expressions (Regex). Η Μεθοδολογία Εντοπισμού (3 Βήματα)
Αναγνώριση Προτύπων (Patterns): Ο κώδικας ψάχνει για τρεις συγκεκριμένες μορφές που κυριαρχούν στα επιστημονικά περιοδικά:Αριθμητικές: [1], [1, 2], [ 93 ] (IEEE/Nature style).Παρενθετικές: (Adams, 2025), (Smith et al., 2020) (APA/Harvard style).Narrative (Αφηγηματικές): Och (2003), Papineni et al. (2002). Σε αυτή τη μορφή, το όνομα είναι έξω από την παρένθεση και η ημερομηνία μέσα.Εντοπισμός του Index:Ο κώδικας βρίσκει τον αριθμό του χαρακτήρα στον οποίο ξεκινάει η πρώτη αναφορά μέσα στην πρόταση. Για παράδειγμα, αν η αναφορά ξεκινάει στον 50ό χαρακτήρα μιας πρότασης 100 χαρακτήρων, ο index είναι 50.Κανονικοποίηση (Normalization):Για να είναι συγκρίσιμες οι θέσεις σε προτάσεις διαφορετικού μήκους, διαιρούμε τον index με το συνολικό μήκος της πρότασης:
$$\text{Relative Position} = \frac{\text{Character Index of Citation}}{\text{Total Sentence Length}}$$
Τιμή κοντά στο 0.0: Η αναφορά είναι στην αρχή.Τιμή κοντά στο 0.5: Η αναφορά είναι στη μέση.Τιμή κοντά στο 1.0: Η αναφορά είναι στο τέλος.

Το df έχει υποστεί επεξεργασία στις sentences. παράδειγμα: [ Welcome Pang and Lee (2004) frame the problem of detecting subjective sentences as finding the minimum cut in a graph representation of the sentences. ==> (Welcome (Pang and Lee, 2004) frame the problem of detecting subjective sentences as finding the minimum cut in a graph representation of the sentences.] προκειμένου να πάρω την σωστή θέση της αναφοράς-πηγής.Αν άφηνα την πρόταση έτσι: [Welcome Pang and Lee (2004)], έπαιρνα για όλη την αναφορά θέση 0. Όμως η λέξη Welcome δεν είναι μέρος της αναφοράς. Οπότε μετατρέποντας την πρόταση σε [Welcome (Pang and Lee, 2004)] ουσιαστικά παίρνω την σωστή θέση της αναφοράς, δηλαδή περίπου 0.089 (τυχαίος αριθμός είναι το 0.089).

In [ ]:
def calculate_citation_position(row):
    # Εφαρμογή μόνο αν is_citation == 1
    if row['is_citation'] != 1:
        return -1  # περίπτωση πρότασης που δεν είναι citation.

    text = str(row['sentence'])

    # --------------------------------------------------
    # 1. Fix LaTeX / OCR escaped brackets
    # --------------------------------------------------
    text = re.sub(r'\\\[', '[', text)
    text = re.sub(r'\\\]', ']', text)

    # --------------------------------------------------
    # 2. Remove remaining backslashes
    # --------------------------------------------------
    text = re.sub(r'\\', '', text)


    # --------------------------------------------------
    # 2. Normalization of narrative / noisy citations
    # --------------------------------------------------
    # Turney (2002) ==> (Turney, 2002)
    text = re.sub(
        r'\b([A-Z][a-zA-Z]+)\s*\(\s*(\d{4})\s*\)',
        r'(\1, \2)',
        text
    )
    # Duygulu et al. , 2002 ==> (Duygulu et al., 2002)
    text = re.sub(
        r'\b([A-Z][a-zA-Z]+(?:\s+et\s+al\.)?)\s*,\s*(\d{4})\b',
        r'(\1, \2)',
        text
    )
    # Smith and Eisner (2006a) ==> (Smith and Eisner, 2006a)
    text = re.sub(
        r'\b([A-Z][a-zA-Z]+(?:\s+(?:and|&)\s+[A-Z][a-zA-Z]+)+)\s*\(\s*(\d{4}[a-z])\s*\)',
        r'(\1, \2)',
        text
    )
    # Leech and Sampson 1987 ==> (Leech and Sampson, 1987)
    text = re.sub(
        r'\b([A-Z][a-zA-Z]+(?:\s+(?:and|&)\s+[A-Z][a-zA-Z]+)+)\s+(\d{4})(?!\d)\b',
        r'(\1, \2)',
        text
    )
    # Adam et al. (2019) ==> (Adam et al., 2019)
    text = re.sub(
        r'\b([A-Z][a-zA-Z]+(?:\s+et\s+al\.)?)\s*\(\s*(\d{4}[a-z]?)\s*\)',
        r'(\1, \2)',
        text
    )


    # --- ΤΟ REGEX ---
    # 1. Αριθμητικές: [1], [1, 2], [ 93 ] -> \[\s*\d+(?:\s*,\s*\d+)*\s*\]
    # 2. Παρενθετικές: (Adams, 2025), (Smith et al., 2020) -> \(\s*[A-Z][a-zA-Z\s\.]*et\s*al\.\s*,\s*\d{4}\s*\)|\(\s*[A-Z][a-zA-Z\s\.]*,\s*\d{4}\s*\)
    # 2. Παρενθετικές: (πλέον πιάνει και & και ;): (Smith & Jones, 2020; Brown, 2022)
    # 3. Αφηγηματικές: Och (2003), Papineni et al. (2002) -> [A-Z][a-zA-Z\s\.]*(?:et\s*al\.\s*)?\(\s*\d{4}\s*\)
    # 4. Narrative: Smith et al. (2020) ή Och (2003)

    combined_pattern = (
        r'\[\s*\d+(?:\s*,\s*\d+)*\s*\]|'  # IEEE: [1], [1,2]
        r'\(\s*[A-Z][^)]*(?:&|and|;)?[^)]*\d{4}[^)]*\)|'  # APA: (Smith & Jones, 2020; Brown, 2022)
        r'\b[A-Z][a-zA-Z\s\.&]*(?:et\s*al\.\s*)?\(\s*\d{4}\s*\)|'  # Narrative: Smith et al. (2020)

        r'\[\s*(?:'
            r'[A-Z][a-zA-Z]+'
            r'(?:\s+(?:and|&)\s+[A-Z][a-zA-Z]+)?'
            r'(?:\s+et\s+al\.)?'
            r'(?:\s*,\s*|\s+)'
            r'\d{4}(?!\d)'
        r')\s*\]'  # Harvard: [Carletta, 1996]

        r'\[\s*(?:'
            r'[A-Z][a-zA-Z]+'                              # Author
            r'(?:\s+(?:and|&)\s+[A-Z][a-zA-Z]+)?'          # and / &
            r'(?:\s+et\s+al\.)?'                           # et al.
            r'\s*,\s*\d{4}'                                # , 2002
            r'(?:\s*;\s*'                                  # ; next citation
                r'[A-Z][a-zA-Z]+'
                r'(?:\s+(?:and|&)\s+[A-Z][a-zA-Z]+)?'
                r'(?:\s+et\s+al\.)?'
                r'\s*,\s*\d{4}'
            r')*'
        r')\s*\]'
    )

    # Εύρεση όλων των εμφανίσεων
    matches = list(re.finditer(combined_pattern, text))

    if not matches:
        return np.nan # περίπτωση αναφοράς -> [1], (Athar, 2020), κλπ. που δεν βρέθηκε.

    # Παίρνουμε την πρώτη αναφορά (συνήθως αυτή ορίζει το context)
    first_match_start = matches[0].start()

    # Υπολογισμός σχετικής θέσης (0.0 έως 1.0)
    relative_pos = first_match_start / len(text)

    return round(relative_pos, 4)

# Εφαρμογή της συνάρτησης
df['Citation_Relative_Position'] = df.apply(calculate_citation_position, axis=1)

Έλεγχος για να δούμε αν υπάρχουν citations που πήραν nan στη στήλη citation_relative_position. Το nan σημαίνει ότι τα regular expression δεν εντόπισαν μοτίβο αναφοράς.

In [ ]:
df_citations = df[df['is_citation'] == 1]
valid_count = df_citations['Citation_Relative_Position'].notna().sum()
nan_count = df_citations['Citation_Relative_Position'].isna().sum()

print(f"Εντοπίστηκαν: {valid_count}")
print(f"NaN: {nan_count} ({(nan_count/len(df_citations))*100:.2f}%)")

In [ ]:
# 4. Υπολογισμός αυτών που πήραν -1 (Θεωρητικά δεν πρέπει να υπάρχουν εδώ αν το is_citation=1)
minus_one_count = (df_citations['Citation_Relative_Position'] == -1).sum()

print("--- ΑΠΟΤΕΛΕΣΜΑΤΑ ΕΛΕΓΧΟΥ ΓΙΑ IS_CITATION = 1 ---")
print(f"Συνολικό πλήθος αναφορών στο Dataset: {len(df_citations)}")
print(f"Εντοπίστηκαν επιτυχώς (Δεκαδική τιμή):  {valid_count}")
print(f"ΑΠΕΤΥΧΑΝ (NaN - Missed by Regex):      {nan_count}")
print(f"Πήραν -1 (Λανθασμένα φίλτρα):           {minus_one_count}")
print(f"Ποσοστό επιτυχίας:                      {(valid_count/len(df_citations))*100:.2f}%")

# 5. ΔΕΙΓΜΑ ΤΩΝ NAN: Ας δούμε τι φταίει
if nan_count > 0:
    print("\n--- ΔΕΙΓΜΑ ΠΡΟΤΑΣΕΩΝ ΠΟΥ ΠΗΡΑΝ NaN (ΓΙΑ ΔΙΟΡΘΩΣΗ) ---")
    missed_examples = df_citations[df_citations['Citation_Relative_Position'].isna()]['sentence'].head(10)
    for i, sent in enumerate(missed_examples):
        print(f"{i+1}. {sent[:150]}...") # Τυπώνουμε τους πρώτους 150 χαρακτήρες

Το feature Multi_Citation_Flag ουσιαστικά μετράει την πολυπλοκότητα των στόχων μέσα στην πρόταση.

0 (Μία ή καμία αναφορά): Το μοντέλο δεν έχει να διαλέξει ανάμεσα σε πολλές πηγές. Ο δρόμος είναι «καθαρός».

1 (Δύο ή περισσότερες αναφορές): Υπάρχει «θόρυβος» και ανταγωνισμός μεταξύ των αναφορών.

Οι γενικές προτάσεις, εφόσον δεν έχουν καμία αναφορά, ανήκουν στην κατηγορία όπου δεν υπάρχει ανταγωνισμός στόχων, άρα το 0 είναι η φυσική τους τιμή.

In [ ]:
def calculate_multi_citation_flag(row):
    if row['is_citation'] != 1:
        return 0

    text = str(row['sentence'])

    citation_count = 0

    # -----------------------------
    # Numeric citations
    # -----------------------------
    numeric_blocks = re.findall(r'\[(.*?)\]', text)
    for block in numeric_blocks:
        nums = re.findall(r'\d+', block)
        citation_count += len(nums)

    # -----------------------------
    # Parenthetical author-year
    # -----------------------------
    parenthetical_blocks = re.findall(r'\(([^)]*\d{4}[^)]*)\)', text)
    for block in parenthetical_blocks:
        parts = re.split(r';', block)
        citation_count += len(parts)

    # -----------------------------
    # Narrative author-year (no parentheses)
    # -----------------------------
    narrative_matches = re.findall(
        r'\b[A-Z][a-z]+(?:\s+(?:and|&)\s+[A-Z][a-z]+)?\s+\d{4}\b',
        text
    )
    citation_count += len(narrative_matches)

    return 1 if citation_count > 1 else 0

# Κλήση calculate_multi_citation_flag


# Δημιουργία στήλης μόνο αν δεν υπάρχει, δίπλα στη στήλη is_citation
if 'Multi_Citation_Flag' not in df.columns:
    df['Multi_Citation_Flag'] = df.apply(calculate_multi_citation_flag, axis=1)

    # Μετακίνηση δίπλα στη στήλη is_citation
    cols = list(df.columns)
    cols.remove('Multi_Citation_Flag')

    insert_pos = cols.index('is_citation') + 1
    cols.insert(insert_pos, 'Multi_Citation_Flag')

    df = df[cols]

    print("Created column: Multi_Citation_Flag")
else:
    print("Column Multi_Citation_Flag already exists — skipped.")

In [ ]:
# ------------------------------------------------
# Έλεγχος ότι υπάρχει ground_truth
# ------------------------------------------------
assert "ground_truth" in df.columns, " Δεν βρέθηκε στήλη ground_truth στο CSV!"

In [ ]:
# ------------------------------------------------
# 3. Εντοπισμός όλων των prediction columns
# ------------------------------------------------
pred_cols = [c for c in df.columns if c.endswith("_pred")]

print("Prediction columns found:")
for c in pred_cols:
    print(" -", c)

In [ ]:
# ------------------------------------------------
# 4. Δημιουργία target columns (correct / incorrect)
# ------------------------------------------------
for col in pred_cols:
    model_name = col.replace("_pred", "")
    correct_col = f"{model_name}_is_correct"

    if correct_col not in df.columns:
        df[correct_col] = (df[col] == df["ground_truth"]).astype(int)
        print(f"Created column: {correct_col}")
    else:
        print(f"Skipped (already exists): {correct_col}")

Δημιουργία στήλης is_narrative. Narrative if it has Author (2020) where Author is NOT inside parentheses.

In [ ]:
# -------------------------
# REGEX PATTERNS
# -------------------------

# Author–year narrative (Smith et al. (2020), Smith and Jones (2019))
author_year_narrative = re.compile(
    r'\b[A-Z][A-Za-z\-]+'
    r'(?:\s+(?:et\s+al\.|and\s+[A-Z][A-Za-z\-]+))?'
    r'\s*\(\s*\d{4}[a-z]?\s*\)'
)

# IEEE numeric citation [35], [12–15], [1,3,5]
bracket_citation = re.compile(
    r'\[\s*\d+(?:\s*[-–]\s*\d+)?(?:\s*,\s*\d+(?:\s*[-–]\s*\d+)?)*\s*\]'
)

# IEEE citation at sentence start → narrative ([35] fails to ...)
leading_ieee = re.compile(
    r'^\s*\[\s*\d+(?:\s*[-–]\s*\d+)?\s*\]'
)

# Author(s) + [IEEE]
author_ieee_narrative = re.compile(
    r'\b[A-Z][A-Za-z\-]+'
    r'(?:\s+(?:and|et\s+al\.)\s+[A-Z][A-Za-z\-]+)?'
    r'\s*\[\s*\d+(?:\s*[-–]\s*\d+)?\s*\]'
)

# -------------------------
# POSITIVE narrative cue phrases
# (ONLY strong cues)
# -------------------------
cue_phrases = [
    "according to",
    "as shown by",
    "as proposed by",
    "as described by",
    "following the work of",
    "using the method of"
]

# -------------------------
# EXPLICIT non-narrative patterns
# (NEVER narrative)
# -------------------------
non_narrative_patterns = [
    r'\bthe findings in\s*\[',
    r'\bthe paper in\s*\[',
    r'\bthe method in\s*\[',
    r'\bthe approach in\s*\[',
    r'\bthe results in\s*\[',
    r'\bthe study in\s*\[',
    r'\bthe model in\s*\['
]

# -------------------------
# FUNCTIONS
# -------------------------

def has_narrative_author_year(text: str) -> bool:
    """
    Smith et al. (2020) argue that ...
    """
    for m in author_year_narrative.finditer(text):
        start = m.start()
        i = start - 1
        while i >= 0 and text[i].isspace():
            i -= 1
        # Exclude cases like "(Smith et al. (2020))"
        if i >= 0 and text[i] == '(':
            continue
        return True
    return False


def has_narrative_ieee(text: str) -> bool:
    lower = text.lower()

    # 0⃣ Explicit exclusions
    for pat in non_narrative_patterns:
        if re.search(pat, lower):
            return False

    # 1⃣ IEEE citation as grammatical SUBJECT
    # "[35] fails to address ..."
    if leading_ieee.search(text):
        return True

    # 2⃣ Author(s) + [number]
    if author_ieee_narrative.search(text):
        return True

    # 3⃣ Strong narrative cue + citation nearby
    for cue in cue_phrases:
        if re.search(rf'\b{cue}\b', lower):
            if bracket_citation.search(text):
                return True

    # 4⃣ Based on ONLY if sentence STARTS with it
    # "Based on [35], we propose ..."
    if re.search(r'^based on\s+(\[|\b[A-Z])', lower):
        if bracket_citation.search(text):
            return True

    return False


def detect_is_narrative(sentence, is_citation):
    if is_citation != 1 or not isinstance(sentence, str):
        return -1

    s = sentence.strip()

    # Author–year narrative
    if has_narrative_author_year(s):
        return 1

    # IEEE narrative
    if has_narrative_ieee(s):
        return 1

    return 0


#if "is_narrative" not in df.columns:
df["is_narrative"] = [
        detect_is_narrative(s, c) for s, c in zip(df["sentence"], df["is_citation"])
]

    # Place is_narrative next to Multi_Citation_Flag
cols = list(df.columns)
cols.remove("is_narrative")
idx = cols.index("Multi_Citation_Flag")
cols = cols[:idx+1] + ["is_narrative"] + cols[idx+1:]
df = df[cols]
#else:
#    print("the column already exists")

In [ ]:
# ------------------------------------------------
# 5. Αποθήκευση ενημερωμένου CSV
# ------------------------------------------------
df.to_csv(MASTER_PATH, index=False, encoding="utf-8")

print(" Target columns (_is_correct) δημιουργήθηκαν και το CSV αποθηκεύτηκε.")

Άθροισμα των μηδενικών (ΛΑΘΗ) για ΟΛΑ τα μοντέλα.

In [ ]:
# Εντοπισμός όλων των στηλών correctness
correct_cols = [c for c in df.columns if c.endswith("_is_correct")]

# Υπολογισμός λαθών (0) για κάθε μοντέλο
wrong_counts = {}

for col in correct_cols:
    model_name = col.replace("_is_correct", "")
    wrong_counts[model_name] = (df[col] == 0).sum()

# Εκτύπωση αποτελεσμάτων
for model, wrong in wrong_counts.items():
    print(f"{model} λάθος προβλέψεις: {wrong}")



Αύξουσα ταξινόμηση μοντέλων βάσει λαθών.

In [ ]:
# Δημιουργία DataFrame από τα λάθη
wrong_df = pd.DataFrame.from_dict(
    wrong_counts, orient="index", columns=["wrong predictions"]
).reset_index().rename(columns={"index": "model"})

# Αύξουσα ταξινόμηση (λιγότερα λάθη πρώτα)
wrong_df_sorted = wrong_df.sort_values(
    by="wrong predictions", ascending=True
)

wrong_df_sorted

Ερώτημα: Πόσα είναι τα δείγματα που έχουν is_citation=1 και Multi_Citation_Flag=0 ??  Θέλουμε τις citations που έχουν μόνο μια πηγή.

In [ ]:
single_citation_count = df[(df['is_citation'] == 1) & (df['Multi_Citation_Flag'] == 0)].shape[0]
print(f"Ο αριθμός των δειγμάτων με is_citation=1 και Multi_Citation_Flag=0 είναι: {single_citation_count}")

Πόσα είναι τα δείγματα που έχουν is_citation=1 και Multi_Citation_Flag=1 ?? Θέλουμε τις citations που έχουν πάνω απο μια πηγή.

In [ ]:
multi_citation_count = df[(df['is_citation'] == 1) & (df['Multi_Citation_Flag'] == 1)].shape[0]
print(f"Ο αριθμός των δειγμάτων με is_citation=1 και Multi_Citation_Flag=1 είναι: {multi_citation_count}")

Πλήθος narrative citations

In [ ]:
narrative_citation_count = df[df['is_narrative'] == 1].shape[0]
print(f"Ο αριθμός των narrative citations είναι: {narrative_citation_count}")

Πλήθος non narrative citations

In [ ]:
non_narrative_citation_count = df[(df['is_citation'] == 1) & (df['is_narrative'] == 0)].shape[0]
print(f"Ο αριθμός των non-narrative citations είναι: {non_narrative_citation_count}")

Εξαγωγή στατικών γλωσσικών χαρακτηριστικών (Sentence_Length, Negation_Count, Semantic_Depth).

In [ ]:
def extract_features(row):
    # Μετατροπή σε string και πεζά για ομοιομορφία
    text = str(row['sentence']).lower()

    # --- FEATURE 1: Sentence Length ---
    # Χωρίζουμε με βάση τα κενά για να βρούμε το πλήθος των λέξεων
    words = text.split()
    sentence_length = len(words)

    # --- FEATURE 2: Negation Count ---
    # Λίστα με λέξεις-αρνήσεις
    negation_words = [
        # Standard Explicit Negators Βασικές αρνήσεις
        'not', 'no', 'never', 'neither', 'nor', 'none',
        # Quasi-negatives Σπάνιες αρνήσεις
        'hardly', 'scarcely', 'barely', 'seldom', 'rarely',
        # Contractions
        'n\'t', 'cannot', 'cant', 'dont', 'doesnt', 'didnt', 'isnt', 'arent', 'wasnt', 'werent', 'hasnt', 'havent',
        # Academic/Scientific Negators Ακαδημαϊκή Αποτυχία
        'lack', 'lacks', 'lacked', 'fail', 'fails', 'failed', 'failure', 'absence', 'without',
        # Επιστημονικοί Περιορισμοί
        'limit', 'limited', 'limitation', 'limitations', 'insufficient', 'poor', 'weak'
    ]

    # Χρήση regex για να βρούμε ολόκληρες λέξεις και την κατάληξη "n't"
    neg_pattern = r"\b(" + "|".join(negation_words) + r")\b|n't\b"
    negations = re.findall(neg_pattern, text)
    negation_count = len(negations)

    # --- FEATURE 3: Semantic Depth (Explicit vs Implicit) ---
    # Ορίζουμε λέξεις-κλειδιά που υποδηλώνουν "φανερό" συναίσθημα
    neg_lexicon = [
        'fail', 'error', 'inconsistent', 'limited', 'disadvantage', 'poor', 'weak',
        'struggle', 'unfortunate', 'drawback', 'lack', 'below', 'shortcoming', 'struggled',
        'insufficient', 'problematic', 'flaw', 'flaws', 'bias', 'biased', 'unclear', 'obscure',
        'incremental', 'difficult'
    ]

    pos_lexicon = [
        'improve', 'outperform', 'excellent', 'robust', 'superior', 'effective',
        'success', 'advantage', 'efficient', 'stable', 'best', 'state-of-the-art', 'sota',
        'novel', 'novelty', 'significant', 'reliable', 'accurate', 'comprehensive', 'validated',
        'validation', 'pioneering', 'contribution'
    ]

    # Default τιμή: 1 (Implicit / Υπαινικτικό)
    depth = 1

    # Αν το ground_truth είναι 0 (Negative) και υπάρχει αρνητική λέξη -> 0 (Explicit)
    if row['ground_truth'] == 0:
        if any(re.search(r'\b' + re.escape(word), text) for word in neg_lexicon):
            depth = 0

    # Αν το ground_truth είναι 2 (Positive) και υπάρχει θετική λέξη -> 0 (Explicit)
    elif row['ground_truth'] == 2:
        if any(re.search(r'\b' + re.escape(word), text) for word in pos_lexicon):
            depth = 0

    # Για το ground_truth 1 (Neutral), το αφήνουμε συνήθως 1 (Implicit)
    # καθώς σπάνια υπάρχουν "έντονες" λέξεις-κλειδιά που το ορίζουν ρητά.

    return pd.Series([sentence_length, negation_count, depth])

# Εφαρμογή της συνάρτησης στο DataFrame
df[['Sentence_Length', 'Negation_Count', 'Semantic_Depth']] = df.apply(extract_features, axis=1)

# Αποθήκευση του νέου αρχείου
df.to_csv('data/processed/Master_results_with_features.csv', index=False)

print("Τα χαρακτηριστικά προστέθηκαν και το αρχείο αποθηκεύτηκε!\n")
print(df[['Sentence_Length', 'Negation_Count', 'Semantic_Depth']].head())

# END